## SCL2205 Preprocessing Notebook
This notebook describes the step-by-step process on the UniProtKB dataset.

### Package imports

In [ ]:
import pandas as pd
import types
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp
import numpy as np
import itertools
from datetime import datetime, date
import re
from collections import Counter
import os

In [ ]:
def imports():
    for name, val in globals().items():
        if isinstance(val, (types.ModuleType)):
            if getattr(val, '__version__', None):
                yield f'{val.__name__}=={val.__version__}'

In [ ]:
list(imports())

### Data dictionary
#### Files
Name|Description
---|---
1_uniprot-dataset-sample-2023.01.24-16.59.27.53.tsv.gz|full uniprot table
2_uniprot-dates-sample-2023.01.24-19.20.55.06.tsv.gz|uniprot dates data
uniprotkb-reviewed-cols-dict_202302.tsv.gz|the shortened column names
scl-mapping-curation.tsv.gz|manual label mapping table
all-uniprot-downloaded-accessions-2023.01.24-16.59.27.53.tsv.gz|the accession of all the samples downloaded from uniprot

### Universal variables

In [ ]:
data = '../data'

In [ ]:
tsv1 = '{data}/1_uniprot-dataset-sample-2023.01.24-16.59.27.53.tsv' # the all-columns reviewed data dump (minus dates)
tsv2 = '{data}/2_uniprot-dates-sample-2023.01.24-19.20.55.06.tsv' # reviewed dates-only dump
dt = datetime.now().strftime('%Y%m%d%H%M')

In [ ]:
# read-in customised fields data
df_tsv1 = pd.read_table(f'{data}/{tsv1}', low_memory=False)
df_tsv1.head(2)

In [ ]:
# read-in dates data
df_tsv2 = pd.read_table(f'{data}/{tsv2}')
df_tsv2.head(2)

### Exploratory analyses

In [ ]:
df_uniprot = df_tsv1.merge(df_tsv2, on='Entry')

In [ ]:
# describe numerical columns
df_uniprot[['Length', 'Mass', 'Sequence version', 'Annotation', 'Entry version']].describe()

In [ ]:
# filter length outliers
df_uniprot.query('Length < 10000')[['Length', 'Mass', 'Sequence version', 'Annotation', 'Entry version']].describe()

In [ ]:
# how many seqs have lengths above mean length
df_uniprot.query('367 < Length <10000').shape[0]

In [ ]:
# visualise columns
sorted_cols = sorted(df_uniprot.columns.to_list())
columns = df_uniprot.columns

In [ ]:
# display columns and number of unique entries per column; indexed-unsorted and sorted
count = 0
cols = []
# for col, scol in zip(columns, sorted_cols):
#     cols.append(col)
#     num_unique = df_uniprot[col].nunique()
#     print(f'{count} {col}: {num_unique}\t\t\t\t{scol}: {num_unique}')
#     count += 1

In [ ]:
# import customised colnames dictionary
df_cols = pd.read_csv(f'{data}/scl-mapping-curation.tsv', sep='\t')
# for row in df_cols.iterrows():
#     print(row[1])

In [ ]:
df_cols.head(2)

In [ ]:
# rename columns
df_uniprot.columns = df_cols.abbr_col_name

### Data filtering

In [ ]:
mask0 = df_uniprot['scl'].isna() # mask for lacking scl
mask1 = df_uniprot.tax_lin.str.contains('Eukaryota (superkingdom)', regex=False) # eukaryotes only
mask2 = df_uniprot.scl.str.contains('ECO:0000269', regex=False) # experimental evidence
mask3 = df_uniprot.tax_lin.str.contains('Bacteria (superkingdom)', regex=False) # prokaryotes only
mask4 = df_uniprot['ann'] >= 3 # above average annotation filter
mask5 = df_uniprot['leng'] >= 30 # alignment length threshold for feature (structure) similarity (Rost1999: https://doi.org/10.1093/protein/12.2.85) 
mask6 = df_uniprot['leng'] <= 5000 # max length

In [ ]:
# subset proteins with subcellular localization (scl) annotations
df_scl_na = df_uniprot[~mask0] # subset entries with scl annotations

# util func for comparing dafaframes after ablation
def compared_df_shapes(before_df: pd.DataFrame, after_df: pd.DataFrame, identifier: str='') -> None:
    shape_b4 = before_df.shape
    shape_af = after_df.shape
    print(f'Original data dimensions: {shape_b4[0]} by {shape_b4[1]}')
    print(f'After {identifier} data dimensions: {shape_af[0]} by {shape_af[1]}')
    print(f'Percent reduction: {round(100 - ((shape_af[0] / shape_b4[0]) * 100), 2)}')
    print(f'Percent left: {round((shape_af[0] / shape_b4[0]) * 100, 2)}')

compared_df_shapes(df_uniprot, df_scl_na)

In [ ]:
# subset proteins with subcellular localization (scl) annotations and belonging to superkingdom eukaryota(2759)
df_scl_ann = df_uniprot[mask1 & ~mask0 & mask2 & mask5] # subset entries with scl annotations
compared_df_shapes(df_uniprot, df_scl_ann)

In [ ]:
# effect of the different filters
df_scl = df_uniprot[~mask0] # subset entries with scl annotations
compared_df_shapes(df_uniprot, df_scl, 'mask0-scl')
print('============================================')
df_scl = df_uniprot[mask1] # subset entries with scl annotations
compared_df_shapes(df_uniprot, df_scl, 'mask1-tax')
print('============================================')
df_scl = df_uniprot[mask2 == True] # subset entries with scl annotations
compared_df_shapes(df_uniprot, df_scl, 'mask2-exp')
print('============================================')
df_scl = df_uniprot[mask4] # subset entries with scl annotations
compared_df_shapes(df_uniprot, df_scl, 'mask4-ann')
print('============================================')
df_scl = df_uniprot[mask5 & mask6] # subset entries with scl annotations
compared_df_shapes(df_uniprot, df_scl, 'mask5/6-len')

In [ ]:
# subset proteins with subcellular localization (scl) annotations and belonging to superkingdom eukaryota(2759)
# mask4: to filter for annotation quality
df_scl = df_uniprot[mask1 & ~mask0 & mask4 & mask2 & mask5 & mask6] # subset entries with scl annotations
compared_df_shapes(df_uniprot, df_scl)

In [ ]:
# export dataframe for entries with scl annotations only
##df_scl.to_csv(f'{data}/uniprotkb-reviewed-complete-scl_{dt}.tsv', sep="\t", index=False)

In [ ]:
# summary statistics
df_scl.describe()

In [ ]:
# export, as df, the scl column value counts
df_scl_col = df_scl['scl'].value_counts().to_frame().reset_index().rename(columns={'index': 'scl'})
# df_scl_col.to_csv(f'{data}/uniprotkb-reviewed-complete-scl-val-cnt_{dt}.tsv', sep="\t")
df_scl_col.head(2)

In [ ]:
# define func to extract scl annotation from dataset column
def extract_scl_ann(string=None, split_point=None, note=True, ann_joiner ='+~', annotations=False, evi_code=None, nummol=False):
    '''This function extracts subcellular localisation annotation from the "Subcellular location [CC]" 
    column of UniProtKB's .tsv download 
    string        the string to split
    split_point   the string that defines the pattern at which to perform the splits
    note          whether to include the note free text in the output
    ann_joiner    a string pattern to be used internally for demarcating annotation entries
    evi_code      for specifying the retrieval of annotations of particular evidence category
    nummol        if True, returns only the number of molecules (isoforms) annotated'''
    import re
    pat = re.compile(r"\[[',.+\/*\[\]()\w -]+\]: ")
    string  = pat.sub('', string) # remove molecule type tags
    if ann_joiner in string:
        print(f'ERROR: The annotation-join string you have chosen ({ann_joiner}) is present in your target string. Please, provide a unique joinner-string')
        return None
    splits = {} # initialise a dictionary for the different levels of splits
    
    if isinstance(split_point, str):
        split = string.split(split_point)
        split = list(map(lambda x: x.strip(), split)) # remove leading/trailing spaces after split
        split = list(filter(lambda x: x != '', split)) # filter non-empty items, if split_point is at the beginning/end of string
        splits[f'Split'] = split
        return splits
    elif isinstance(split_point, list):
        level = 1
        for i, point in enumerate(split_point):
            if i == 0 and point in string:
                split = string.split(point)
                split = list(map(lambda x: f'//{x}', split)) # add marker (//) for distinguishing protein molecules (types/isoforms)
                split = list(filter(lambda x: x != '//', split)) # filter non-marker-only items, if split_point is at the beginning/end of string
                splits[f'level_{level}'] = split # add annotations as levels correspnding to # split points
            elif i > 0  and point in string:
                level += 1 # increament level
                to_split_prev = splits[f'level_{level - 1}'] # -1 targets the members of the previous round of splitting
                split = list(map(lambda x: x.split(point), to_split_prev)) # broaddcast the split to all members of the previous split. Results in list of lists
                next_lvl = list(itertools.chain.from_iterable(split)) #  concatanate the split list above
                next_lvl = list(filter(lambda x: x != '', next_lvl)) # filter non-empty items, if split_point is at the beginning/end of string 
                
                # do we want to include the Note section in level_2 and onwards results?
                if not note and len(next_lvl) > 1:
                    next_lvl = list(filter(lambda x: 'Note=' not in x, next_lvl)) # filter out items lacking the note tag
                splits[f'level_{level}'] = next_lvl
            else:
                error = f'Separator "{point}" not in obj. Split cannot be done on it, skipped'
                # print(f'{"=" * len(error)}\n{error}\n{"=" * len(error)}')
        to_ext = splits[f'level_{level}']
        to_ext_str = ann_joiner.join(to_ext) # join members of level. The separator here will help to distinguish different annotation entries
        num_mol = len(to_ext_str.split('//')) - 1 # since // is leading there will be an empty list item after split
        
        # if we want number of SCL for particular evidence code
        if nummol and evi_code:
            num_mol = len(list(filter(lambda x: evi_code in x, to_ext_str.split('//'))))
            return num_mol
        elif nummol:
            annotations, evi_code = False, None
            return num_mol 
        
        # if we want only the actual SCL and not the split-tree
        if annotations == True:
            def get_ann_from_list(ann_list):
                    scl_ann = list(map(lambda x: x.strip().lstrip('//').split(' {')[0], ann_list))
                    scl_ann = ';'.join(scl_ann).replace(', ', ';').replace('. ', ';') # account for SCL separated by full-stop
                    _ = scl_ann.split(';') # to capitalise each SCL since the annotations are sentence-cased
                    return ';'.join(sorted(set(map(lambda x: x.strip().capitalize(), _))))
            
            if evi_code and evi_code in to_ext_str:
                ann_list = list(filter(lambda x: evi_code in x.strip(), to_ext))
                prot_scl = get_ann_from_list(ann_list)
                
                if prot_scl == '':
                     return ' '.join((evi_code, 'in Note')) # evidence was only seen in the note-tag free-text   
                return prot_scl
            else:
                prot_scl = get_ann_from_list(to_ext)
                
                if prot_scl == '':
                     return 'No Evidence' # lacking evidence or something else
                return prot_scl
        elif annotations == False and evi_code:
            splits[f'level_{level}'] = list(filter(lambda x: evi_code in x, splits[f'level_{level}'])) # return split-tree for particular evidence code
 
        return splits
    else:
        print('Something is wrong')
            

In [ ]:
#func test examples
samp_ann1 = 'SUBCELLULAR LOCATION: Endomembrane system {ECO:0000269|PubMed:11744688}. Synapse, synaptosome {ECO:0000250|UniProtKB:Q9WVE9}. Cell projection, lamellipodium {ECO:0000269|PubMed:11744688}. Cell membrane {ECO:0000269|PubMed:11744688, ECO:0000269|PubMed:20946875}. Membrane, clathrin-coated pit {ECO:0000269|PubMed:20946875, ECO:0000269|PubMed:29887380}. Recycling endosome {ECO:0000269|PubMed:29030480}. Endosome {ECO:0000250|UniProtKB:Q9Z0R4}. Cytoplasmic vesicle {ECO:0000250|UniProtKB:Q9Z0R4}. Note=Colocalizes with SGIP1 at the plasma membrane in structures corresponding most probably to clathrin-coated pits (PubMed:20946875). Colocalizes with RAB13 on cytoplasmic vesicles that are most likely recycling endosomes (PubMed:29030480). {ECO:0000269|PubMed:20946875, ECO:0000269|PubMed:29030480}.; SUBCELLULAR LOCATION: [Isoform 2]: Cytoplasm {ECO:0000269|PubMed:29599122}. Endomembrane system {ECO:0000269|PubMed:21712076}. Nucleus envelope {ECO:0000269|PubMed:29599122}. Note=Shuttles between the cytoplasm and nucleus in an XPO1/CRM1-dependent manner. {ECO:0000269|PubMed:29599122}.; SUBCELLULAR LOCATION: [Isoform 5]: Endomembrane system {ECO:0000269|PubMed:21712076}.'
samp_ann2 = 'SUBCELLULAR LOCATION: Cytoplasm {ECO:0000269|PubMed:15647271}. Nucleus {ECO:0000269|PubMed:15647271, ECO:0000269|PubMed:22781750}. Note=Cytoplasmic in the absence of ligand. Migrates to the nucleus when complexed with SMAD4 (PubMed:15647271). Co-localizes with LEMD3 at the nucleus inner membrane (PubMed:15647271). Exported from the nucleus to the cytoplasm when dephosphorylated (By similarity). {ECO:0000250|UniProtKB:P70340, ECO:0000269|PubMed:15647271}.'
samp_ann3 = 'SUBCELLULAR LOCATION: Secreted. Fimbrium. Note=At the tip of P pili. {ECO:0000269|PubMed:2886993}.'
samp_ann4 = '''SUBCELLULAR LOCATION: Cell membrane {ECO:0000269|PubMed:21436032, ECO:0000269|PubMed:23217710}; Single-pass membrane protein {ECO:0000269|PubMed:21436032, ECO:0000269|PubMed:23217710}. 
Note=Efficient localization to the plasma membrane requires the presence of LHFPL5.; SUBCELLULAR LOCATION: [Isoform 1]: Cell membrane {ECO:0000250}; Single-pass type I membrane protein {ECO:0000250}.; 
SUBCELLULAR LOCATION: [Isoform 2]: Cell membrane {ECO:0000250}; Single-pass type I membrane protein {ECO:0000250}.; SUBCELLULAR LOCATION: [Isoform 3]: Secreted {ECO:0000305}.; SUBCELLULAR LOCATION: 
[Isoform 4]: Cell membrane {ECO:0000250}; Single-pass type I membrane protein {ECO:0000250}.; SUBCELLULAR LOCATION: [Isoform 5]: Cell membrane {ECO:0000250}; Single-pass type I membrane protein 
{ECO:0000250}.; SUBCELLULAR LOCATION: [Isoform 6]: Cell membrane {ECO:0000250}; Single-pass type I membrane protein {ECO:0000250}.; SUBCELLULAR LOCATION: [Isoform 7]: Cell membrane {ECO:0000250}; 
Single-pass type I membrane protein {ECO:0000250}.; SUBCELLULAR LOCATION: [Isoform 8]: Cell membrane {ECO:0000250}; Single-pass type I membrane protein {ECO:0000250}.; SUBCELLULAR LOCATION: [Isoform 9]: 
Cell membrane {ECO:0000250}; Single-pass type I membrane protein {ECO:0000250}.; SUBCELLULAR LOCATION: [Isoform 10]: Cell membrane {ECO:0000250}; Single-pass type I membrane protein {ECO:0000250}.; 
SUBCELLULAR LOCATION: [Isoform 11]: Cell membrane {ECO:0000250}; Single-pass type I membrane protein {ECO:0000250}.; SUBCELLULAR LOCATION: [Isoform 12]: Cell membrane {ECO:0000250}; Single-pass type I 
membrane protein {ECO:0000250}.; SUBCELLULAR LOCATION: [Isoform 13]: Secreted {ECO:0000305}.; SUBCELLULAR LOCATION: [Isoform 14]: Secreted {ECO:0000305}.; SUBCELLULAR LOCATION: [Isoform 15]: Secreted 
{ECO:0000305}.; SUBCELLULAR LOCATION: [Isoform 16]: Secreted {ECO:0000305}.; SUBCELLULAR LOCATION: [Isoform 17]: Secreted {ECO:0000305}.; SUBCELLULAR LOCATION: [Isoform 18]: CelSep 14, 2023l membrane {ECO:0000250}; 
Single-pass type I membrane protein {ECO:0000250}.; SUBCELLULAR LOCATION: [Isoform 19]: Cell membrane {ECO:0000250}; Single-pass type I membrane protein {ECO:0000250}.; SUBCELLULAR LOCATION: [Isoform 21]: 
Secreted {ECO:0000305}.; SUBCELLULAR LOCATION: [Isoform 22]: Secreted {ECO:0000305}.; SUBCELLULAR LOCATION: [Isoform 23]: Secreted {'ECO:0000305'}.'''

In [ ]:
extract_scl_ann(samp_ann4, ['SUBCELLULAR LOCATION: ', '}. '], note=False, ann_joiner='~', annotations=False, evi_code=None, nummol=True)

In [ ]:
extract_scl_ann(samp_ann4, ['SUBCELLULAR LOCATION: ', '}. '], note=False, ann_joiner='~', annotations=False, evi_code='ECO:0000269', nummol=True)

In [ ]:
extract_scl_ann(samp_ann4, ['SUBCELLULAR LOCATION: ', '}. '], note=False, ann_joiner='~', annotations=True, evi_code='ECO:0000269', nummol=False)

In [ ]:
extract_scl_ann(samp_ann1, ['SUBCELLULAR LOCATION: ', '}. '], note=True, ann_joiner='~', annotations=True, evi_code=None)

In [ ]:
extract_scl_ann(samp_ann1, ['SUBCELLULAR LOCATION: ', '}. '], note=True, ann_joiner='~', annotations=True, evi_code=None)

In [ ]:
extract_scl_ann(samp_ann1, ['SUBCELLULAR LOCATION: ', '}. '], note=True, ann_joiner='~', annotations=False, evi_code='ECO:0000250')

In [ ]:
extract_scl_ann(samp_ann1, ['SUBCELLULAR LOCATION: ', '}. '], note=True, ann_joiner='~', annotations=True, evi_code='ECO:0000269')

In [ ]:
scl_ann = df_scl['scl'].apply(extract_scl_ann, split_point=['SUBCELLULAR LOCATION: ', '}. '], note=False, ann_joiner ='+~', annotations=True, evi_code='ECO:0000269')

In [ ]:
# subset for fields of interest in the entries containing the experimental evidence code (df_scl)
fields_oi = ['entry', 'ent_name', 'gen_name_pri', 'org', 'org_id', 'tax_lin', 'leng',
             'mass', 'seq', 'ph_dep', 'ann', 'feat', 'scl', 'sig_pep', 'beta',
             'turn', 'helix', 'comp_bias', 'dom_ft', 'reg', 'rep', 'pmid', 'dt_creat']#[0, 2, 4, 5, 11, 21, 29, 40, 50,57, 78, 91, 94, 95, 96, 99, 102, 103, 104]
df_field_oi = df_scl.loc[:,fields_oi]
df_field_oi.columns

In [ ]:
# add extracted annotation column and number of molecules
sr_scl = df_scl['scl'].apply(extract_scl_ann, split_point=['SUBCELLULAR LOCATION: ', '}. '], note=False, ann_joiner ='+~', annotations=True, evi_code='ECO:0000269')
sr_scl_mol = df_scl['scl'].apply(extract_scl_ann, split_point=['SUBCELLULAR LOCATION: ', '}. '], note=False, evi_code='ECO:0000269', nummol=True)
df_field_scl = df_field_oi.assign(scl_ext=sr_scl, scl_num_mol=sr_scl_mol)[[
    'entry', 'ent_name', 'gen_name_pri', 'org', 'org_id', 'tax_lin', 'leng',
    'mass', 'seq', 'ph_dep', 'ann', 'feat', 'scl_ext', 'scl',
    'scl_num_mol', 'sig_pep', 'beta', 'turn', 'helix', 'comp_bias',
    'dom_ft', 'reg', 'rep', 'pmid', 'dt_creat']] # reordering inserted columns
df_scl_not_curated = df_field_scl

In [ ]:
# include manually curated (mapped) SCL
df_man_cur_scl = pd.read_csv(f'{data}/scl-mapping-curation.tsv', sep='\t')
df_man_cur_scl.head()
df_man_cur = df_man_cur_scl[~df_man_cur_scl.abbr_col_name.isna()]
dict_man_cur = dict(zip(df_man_cur.org_col_name, df_man_cur.abbr_col_name))

In [ ]:
# add new column of manually curated scl
df_field_scl.insert(loc=12, column='scl_curated', value=df_field_scl.scl_ext.replace(dict_man_cur)) # insert at specific location

# capture the effect of manual curation
df_field_scl_all = df_field_scl.copy()

# subset where counts for scl is at least 100 occurrences
df_field_scl = df_field_scl[df_field_scl.groupby('scl_curated').transform('size') >= 100]

### At this point *`df_field_scl`* is filtered: na, euka, ann3, len>30<500; scl manually curated and cnt100.

In [ ]:
# # saving preprocessed data
# df_field_scl.to_csv(f'{data}/uniprotkb-scl-preprocessed_202308.tsv', sep="\t", index=False)

### Further EDA

In [ ]:
df_field_scl_all.columns

In [ ]:
# scl extracted annotations values count summary
df_field_scl_all.scl_ext.value_counts().size

In [ ]:
# scl curated annotations values count summary
df_field_scl_all.scl_curated.value_counts().size

In [ ]:
# scl curated annotations over 100 samples values count summary
df_field_scl.scl_curated.value_counts().size

In [ ]:
# scl curated annotations over 100 samples values count summary
df_field_scl.scl_curated.value_counts()

In [ ]:
# check how manual curation has affected counts
df = df_field_scl
df.head(2)

In [ ]:
# to check how manual curation has affected counts
df2 = df_scl_not_curated[df_scl_not_curated.groupby('scl_ext').transform('size') >= 100]
df2.head(2)

In [ ]:
sr_counts_b4 = df_field_scl_all.scl_ext.value_counts().to_frame()
sr_counts_4b = df_field_scl.scl_curated.value_counts().to_frame()

df_scl_b4_4b = sr_counts_b4.merge(sr_counts_4b, how='right', left_index=True, right_index=True) # (Endoplasmic reticulum) ER initially 150
df_scl_b4_4b

In [ ]:
caption = '''
Table summarising the label (localisation annotation) counts before and after manual curation to minimise overlap.
Only single-category labels are represented. 
Two labels initially having counts below 100 examples, Plastid and Cell projection, increased by 103.67% and 32.33% respectively.
The two would be omitted otherwise, leading to reduced training diversity. 
'''
df_curation_comp = df_scl_b4_4b.reset_index().rename(columns={'scl_curated': 'Label', 'count_x': 'Before Curation', 'count_y': 'After Curation'}).fillna(150).astype({'Before Curation': int, 'After Curation': int})

df_curation_comp['Percent Increase (%)'] = round((df_curation_comp['After Curation'] / df_curation_comp['Before Curation']), 2)

# df_cur_comp_per = 
df_curation_comp

In [ ]:
print(f"{df_curation_comp.to_latex(index=False, float_format='%.2f', escape=True, multicolumn=False, caption=caption, label='tab:sct')}")

In [ ]:
# review longer sequences
##df_field_scl[df.leng > 5000][['entry', 'scl_ext', 'scl_curated', 'scl', 'leng']].to_csv(f'{data}/uniprotkb-reviewed-over5k-len-scl-ext_{dt}.csv', sep=',', index=False)

In [ ]:
# median seq length of curated scls
df.pivot_table(
values='leng',
columns='scl_curated',
aggfunc=['median'])

In [ ]:
# median seq length of uncurated scls
df2.pivot_table(
values='leng',
columns='scl_ext',
aggfunc=['median'])

### Distibution of sequence lengths across targets

In [ ]:
# group by scl_curated and plot sequence length distribution
grps = df_field_scl[['entry', 'scl_curated', 'leng']].groupby('scl_curated')
lens_dict = {grp[0]:list(grp[1]['leng']) for grp in grps}
lens_dict = dict(sorted(lens_dict.items(), key=lambda x: np.median(x[1])))
sns.boxplot(list(lens_dict.values()))
plt.xticks(ticks=range(28), labels=lens_dict.keys(), rotation=90)
plt.show()

In [ ]:
# group by scl_ext and plot sequence length distribution
grps = df2[['entry', 'scl_ext', 'leng']].groupby('scl_ext')
lens_dict2 = {grp[0]:list(grp[1]['leng']) for grp in grps}
lens_dict2 = dict(sorted(lens_dict2.items(), key=lambda x: np.median(x[1])))
sns.boxplot(list(lens_dict2.values()))
plt.xticks(ticks=range(28), labels=lens_dict2.keys(), rotation=90)
plt.show()

In [ ]:
# determine outlier threshold for each scl_ext
grps = df2[['entry', 'scl_ext', 'leng']].groupby('scl_ext')
outlier_dict2 = {grp[0]:np.quantile(grp[1]['leng'], .75, method='midpoint') + sp.stats.iqr(grp[1]['leng']) * 1.5 for grp in grps}
outlier_dict2 = dict(sorted(outlier_dict2.items(), key=lambda x: x[1]))
outlier_dict2

In [ ]:
# determine outlier threshold for each scl_curated
grps = df_field_scl[['entry', 'scl_curated', 'leng']].groupby('scl_curated')
outlier_dict = {grp[0]:np.quantile(grp[1]['leng'], .75, method='midpoint') + sp.stats.iqr(grp[1]['leng']) * 1.5 for grp in grps}
outlier_dict = dict(sorted(outlier_dict.items(), key=lambda x: x[1]))
outlier_dict

In [ ]:
median_outlier_thresh = np.median(list(outlier_dict.values()))
median_outlier_thresh

In [ ]:
# entries over threshold
ent_overthesh = []
for grp in grps:
    ents = grp[1][grp[1].leng > outlier_dict[grp[0]]].entry
    ent_overthesh += list(ents)
    # print(list(ents))est
# ent_overthesh   

In [ ]:
median_outliers_len = df_field_scl[df_field_scl.entry.isin(ent_overthesh)].leng.median()
median_outliers_len

In [ ]:
median_outlier_thresh + median_outliers_len

In [ ]:
len(lens_dict)

In [ ]:
fig, axs = plt.subplots(28,1,figsize=(10,35))
index = 0
# ax[0]
for n,l in lens_dict.items():
    sns.histplot(l, ax=axs[index]).axvline(np.median(l), ls='--', color='red')
    axs[index].set_title(n)
    axs[index].annotate(f'median: {np.median(l)}', 
                       xy=(np.median(l),.8), #max(ax[index].get_yticks())/2)
                       xytext=(np.median(l) + 100,.8),
                      xycoords=("data", "axes fraction"),
                       color='red')
    index += 1
fig.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(28,1,figsize=(10,35))
index = 0
# ax[0]
for n,l in lens_dict2.items():
    sns.histplot(l, ax=axs[index]).axvline(np.median(l), ls='--', color='red')
    axs[index].set_title(n)
    axs[index].annotate(f'median: {np.median(l)}', 
                       xy=(np.median(l),.8), #max(ax[index].get_yticks())/2)
                       xytext=(np.median(l) + 100,.8),
                      xycoords=("data", "axes fraction"),
                       color='red')
    index += 1
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df_field_scl.leng, ax=ax, kde=True).axvline(np.median(df_field_scl.leng), ls='--', color='orange')
# ax.set_yscale('log')
# ax.set_xscale('log')
ax.annotate(f'median: {np.median(df_field_scl.leng)}', 
                   xy=(np.median(df_field_scl.leng),.8),
                   xytext=(np.median(df_field_scl.leng) + 100,.8),
                  xycoords=("data", "axes fraction"),
                   color='orange')
ax.annotate(f'Summary statistics:\n{df_field_scl.leng.describe()}', 
                   xy=(.5,.25),
                   xytext=(.5,.25),
                  xycoords=("axes fraction", "axes fraction"),
                   color='blue')
fig.tight_layout()
plt.show()

In [ ]:
sns.displot(df_field_scl.leng, log_scale=True)
ax.annotate(f'median: {np.median(df_field_scl.leng)}', 
                   xy=(np.median(df_field_scl.leng),.8),
                   xytext=(np.median(df_field_scl.leng) + 100,.8),
                  xycoords=("data", "axes fraction"),
                   color='orange')
ax.annotate(f'Summary statistics:\n{df_field_scl.leng.describe()}', 
                   xy=(.5,.8),
                   xytext=(.5,.8),
                  xycoords=("axes fraction", "axes fraction"),
                   color='blue')
plt.show()

In [ ]:
g = sns.displot(df_field_scl.leng, log_scale=False)

g.tight_layout()
plt.show()

In [ ]:
df_field_scl.leng.describe()

In [ ]:
# export scl counts
# df_field_scl.scl_curated.value_counts().to_frame().reset_index().to_csv(
    # f'{data}/uniprotkb-scl-preprocessed-class-cnt_{dt}.tsv', sep='\t', index=False)

In [ ]:
# sort by date of seq creation to id earliest publications
df_dt = (df_field_scl.
sort_values(by=['dt_creat', 'pmid'])[['dt_creat', 'scl', 'ann', 'pmid']].
dropna(subset='pmid'))
##df_dt.to_csv(f'{data}/uniprotkb-reviewed-interest-scl-pubs_{dt}.tsv', sep='\t', index=False)

In [ ]:
# scl annotations values count summary
df_field_scl.scl_ext.value_counts()[:20]

In [ ]:
# scl annotations values count summary
df_field_scl.scl_curated.value_counts()[:20]

In [ ]:
# annotations quality values count summary
df_field_scl.ann.value_counts()

In [ ]:
# df_field_scl.scl_ext.value_counts().to_frame().reset_index().rename(columns={'index': 'scl', 'scl_ext': 'counts'}).to_csv(
#     f'{data}/uniprotkb-reviewed-interest-scl-expt-evi-cnt_{dt}.tsv', sep='\t', index=False)

In [ ]:
# summarise organisms
df_field_scl.org.str.contains('Viridiplantae').value_counts()#[:5]

In [ ]:
# summarise taxonomy
df_field_scl.tax_lin.str.contains('Opisthokonta (no rank)', regex=False).value_counts()#[:5]

In [ ]:
# summarise taxonomy
df_field_scl.tax_lin.str.contains('Viridiplantae (kingdom)', regex=False).value_counts()#[:5]

In [ ]:
def remove_multilabels(df: pd.DataFrame) -> pd.DataFrame:
    multilabel = ['Centrosome;Cytoplasm;Cytoskeleton;Microtubule organizing center', 
                  'Cytoplasm;Cytoskeleton', 'Cytoplasm;Membrane', 'Cytoplasm;Nucleus']
    return df[~df.scl_curated.isin(multilabel)]

In [ ]:
# summarise taxonomy: multilabel included
tax_counter_multi = {}
tax_counter_multi['Other'] = 0
no_king_tax_multi = []

for l in df_field_scl.tax_lin.str.split(','):
    try:
        tax = list(filter(lambda x: '(kingdom)' in x, l))[0].strip()
        if tax in tax_counter_multi:
            tax_counter_multi[tax] += 1
        else:
            tax_counter_multi[tax] = 1
    except IndexError:
        no_king_tax_multi.append(l)
        tax_counter_multi['Other'] += 1
        pass
# df_field_scl.tax_lin.str.split(',')

In [ ]:
# summarise taxonomy: multilabel omitted
tax_counter = {}
tax_counter['Other'] = 0
no_king_tax = []

for l in remove_multilabels(df_field_scl).tax_lin.str.split(','):
    try:
        tax = list(filter(lambda x: '(kingdom)' in x, l))[0].strip()
        if tax in tax_counter:
            tax_counter[tax] += 1
        else:
            tax_counter[tax] = 1
    except IndexError:
        no_king_tax.append(l)
        tax_counter['Other'] += 1
        pass
# df_field_scl.tax_lin.str.split(',')

In [ ]:
tax_counter_multi

In [ ]:
tax_counter

In [ ]:
no_king_tax[:3]

### Generate fasta for the proteins

In [ ]:
# pd.read_csv(f'{data}/uniprotkb-scl-preprocessed_202308.tsv', sep='\t')
df_data = (df_field_scl[['entry', 'ent_name', 'seq', 'leng']]) # .sort_values('leng', ascending=False) # for CD-HIT

In [ ]:
df_data.shape[0]

In [ ]:
# generate fasta files
fastas = df_data.entry + ':' + df_data.ent_name + f'\n{df_data.seq}\n'
# fastas.head()
with open(f'{data}/uniprotkb-scl-preprocessed_{dt}.fasta', 'w') as f:
    for fasta in df_data.iterrows():
        f.writelines(f'>{fasta[1][0]}' + ':' + fasta[1][1] + f'\n{fasta[1][2]}\n')

In [ ]:
# survey potentially cleaved AAs
df_data[~df_data.seq.str.startswith('M')].shape[0]

In [ ]:
# survey ambiguous AAs
df_data[~df_data.seq.str.contains(r'X|B|Z|J')].shape[0]